# NeuroPhenotype-ML — Colab bootstrap + first reproducible run
This notebook connects to a **private** GitHub repository without putting a token into code or output, mounts Google Drive, performs a synthetic feature smoke test, and logs one infrastructure run. **It does not download real neuronal recordings or train a model.**

**One-time setup:** In Colab, open the left-side **Secrets** (key icon) → add `GITHUB_TOKEN` → enable notebook access. Use a fine-grained GitHub personal access token with **Contents: Read-only** for **only** `Noel-Betelgeuse/NeuroPhenotype-ML`. Do not share or print it. Tokens are not uploaded to Drive.


In [ ]:
# 1. Authenticated archive download into the ephemeral Colab runtime
from google.colab import userdata
from pathlib import Path
import os, requests, subprocess, sys, tempfile, zipfile
token = userdata.get('GITHUB_TOKEN')
if not token: raise RuntimeError('Add GITHUB_TOKEN in Colab Secrets and enable notebook access.')
api = 'https://api.github.com/repos/Noel-Betelgeuse/NeuroPhenotype-ML'
headers = {'Authorization': f'Bearer {token}', 'Accept': 'application/vnd.github+json', 'X-GitHub-Api-Version': '2022-11-28'}
response = requests.get(api + '/commits/main', headers=headers, timeout=45)
response.raise_for_status()
commit = response.json()['sha']
archive = requests.get(api + '/zipball/' + commit, headers=headers, timeout=120)
archive.raise_for_status()
work = Path('/content/neurophenotype_checkout')
work.mkdir(parents=True, exist_ok=True)
archive_path = work / 'source.zip'
archive_path.write_bytes(archive.content)
with zipfile.ZipFile(archive_path) as z:
    z.extractall(work)
dirs = [p for p in work.iterdir() if p.is_dir() and (p / 'pyproject.toml').exists()]
if len(dirs) != 1: raise RuntimeError('Unexpected repository archive layout')
repo_dir = dirs[0]
del token, headers, response, archive
print('Repository loaded securely; revision:', commit[:12])
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo_dir)], check=True)


In [ ]:
# 2. Attach Drive and verify existing research storage (no duplicate folder creation)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/NeuroPhenotype-ML')
if not DRIVE_ROOT.is_dir():
    raise FileNotFoundError('NeuroPhenotype-ML not under MyDrive. Check folder ownership, mount, shortcuts or Shared drives before proceeding.')
RUNS = DRIVE_ROOT / '04_EXPERIMENTS' / 'RUNS'
if not RUNS.is_dir():
    raise FileNotFoundError('Expected 04_EXPERIMENTS/RUNS does not exist. Check the selected Drive root.')
print('Existing Drive storage verified:', DRIVE_ROOT.name)


In [ ]:
# 3. Check hardware and run package smoke test (synthetic data only)
import platform, importlib, json, time
from neurophenotype.quickcheck import main as quickcheck
quickcheck()
gpu_name = None
try:
    import torch
    if torch.cuda.is_available(): gpu_name = torch.cuda.get_device_name(0)
except ImportError:
    pass
print('Python:', platform.python_version(), '| GPU:', gpu_name or 'not detected')


In [ ]:
# 4. Create a reproducible infrastructure run and export with checksum verification
from neurophenotype.experiment_registry import create_run, export_file_verified
LOCAL_RUNS = Path('/content/neurophenotype_runs')
run_id, metadata_path = create_run(
    LOCAL_RUNS, project='NeuroPhenotype-ML', code_commit=commit,
    details={'python_version': platform.python_version(), 'gpu_name': gpu_name, 'source': 'Colab', 'synthetic_smoke_test': 'passed'},
)
destination = RUNS / run_id / 'run.json'
digest = export_file_verified(metadata_path, destination)
print('Saved run:', run_id)
print('Drive path:', destination)
print('SHA256:', digest)
print('No model was trained.')
